# Flair Feature-Family Target Multitask Training

Train a Flair native multitask model from the Quant Warehouse FMP feature-family/target matrix.

Shape:

- Universe: FMP common equities with `market_cap >= 1_000_000_000_000`.
- Features: one text row per `(date, symbol, feature_family, target_task)`.
- Text: key-value pairs for feature values only, with numeric values rounded to two decimals. `symbol` and `date` are kept as metadata columns and are not included in text.
- Targets: oracle trades, congress buy/sell, insider buy/sell, analyst upgrades/downgrades, price targets, guidance, and earnings are separate Flair classification tasks.

This notebook intentionally does not run anything at import time beyond configuration. Execute cells manually when you are ready to train.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
from time import perf_counter
import math
import random
import re
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "quant_orchestrator").exists():
            return candidate
    raise RuntimeError(f"Could not find quant-orchestrator repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore", category=UserWarning)

import torch
import flair
from flair.data import Corpus, Sentence
from flair.embeddings import TransformerDocumentEmbeddings
from flair.models import MultitaskModel, TextClassifier
from flair.trainers import ModelTrainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    combine_target_panels,
    evaluate_feature_target_matrix,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
    summarize_binary_targets,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

In [ ]:
RANDOM_SEED = 20260630
MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = "2018-01-01"
END_DATE = None
TRAIN_END = pd.Timestamp("2023-12-31")
DEV_END = pd.Timestamp("2024-12-31")

MAX_FEATURES_PER_FAMILY = 50
MIN_ROWS_PER_TASK = 120
MIN_POSITIVE_ROWS_PER_TASK = 10
MIN_FEATURE_COVERAGE = 0.50
MAX_ROWS_PER_TASK_SPLIT = None  # set to an int such as 5_000 for a smoke training run

FLAIR_TRANSFORMER = "prajjwal1/bert-tiny"
FLAIR_EPOCHS = 1
FLAIR_BATCH_SIZE = 16
FLAIR_BATCH_CHUNK_SIZE = 8
FLAIR_EVAL_BATCH_SIZE = 32
FLAIR_LEARNING_RATE = 1e-4
FINE_TUNE_TRANSFORMER = False

ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebooks" / "mult-ml-frameworks" / "flair_feature_family_target_mtl"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
flair.device = TORCH_DEVICE

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=START_DATE,
    end_date=END_DATE,
    horizons=(20, 60),
    min_observations=MIN_ROWS_PER_TASK,
    max_features_per_family=MAX_FEATURES_PER_FAMILY,
)

TARGET_CONFIG = BinaryTargetConfig(
    provider="fmp",
    start_date=FEATURE_CONFIG.start_date,
    end_date=FEATURE_CONFIG.end_date,
    event_families=(
        "congress",
        "insider",
        "analyst_rating",
        "price_target",
        "guidance",
        "earnings",
    ),
    oracle_trade_k_by_frequency={"YE": tuple(range(1, 13))},
    oracle_trade_min_profit_pct=0.01,
    oracle_trade_long_entry_price_col="high",
    oracle_trade_long_exit_price_col="low",
    oracle_trade_short_entry_price_col="low",
    oracle_trade_short_exit_price_col="high",
)

runtime_config = pd.DataFrame(
    [
        {"setting": "min_market_cap", "value": f"{MIN_MARKET_CAP:,}"},
        {"setting": "date_range", "value": f"{START_DATE} to {END_DATE or 'latest warehouse row'}"},
        {"setting": "max_features_per_family", "value": MAX_FEATURES_PER_FAMILY},
        {"setting": "event_families", "value": ", ".join(TARGET_CONFIG.event_families)},
        {"setting": "event_target_rule", "value": "same-day event labels only"},
        {"setting": "oracle_trade_k", "value": str(TARGET_CONFIG.oracle_trade_k_by_frequency)},
        {"setting": "text_rule", "value": "feature key-value pairs only; numeric values rounded to 2 decimals; no symbol/date in text"},
        {"setting": "torch_device", "value": str(TORCH_DEVICE)},
        {"setting": "flair_transformer", "value": FLAIR_TRANSFORMER},
    ]
)
display(runtime_config)

## Build the Feature/Target Matrix

This mirrors the Quant Warehouse `feature_family_binary_target_matrix` notebook and keeps the `1T+` FMP universe. The output is still tabular at this point; text rows are created in the next section.

In [ ]:
started = perf_counter()

WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(
    config=WAREHOUSE.config,
    backend=WAREHOUSE.backend,
    catalog=WAREHOUSE.catalog,
    fundamentals=WAREHOUSE.fundamentals,
    equity_calendar=WAREHOUSE.equity_calendar,
)

symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(FEATURE_CONFIG, warehouse=WAREHOUSE)

print(f"Universe source: {universe_source}")
print(f"Eligible symbols: {len(symbols)}")
print(symbols)

display(eligibility.head(30))

In [ ]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols,
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)

print(
    {
        "screened_symbols": len(symbols),
        "raw_feature_panel_rows": len(raw_feature_panel),
        "raw_features": raw_feature_metadata["feature"].nunique(),
        **{key: round(value, 3) for key, value in feature_timings.items()},
    }
)

display(feature_diagnostics.sort_values(["status", "rows"], ascending=[True, False]).head(30))

In [ ]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols,
    TARGET_CONFIG,
    event_store=EVENT_STORE,
    include_historical=True,
)

event_symbols = tuple(
    event_diagnostics.loc[event_diagnostics["combined_rows"].gt(0), "symbol"].sort_values().tolist()
)
feature_panel = raw_feature_panel.loc[raw_feature_panel["symbol"].isin(event_symbols)].copy()

selected_features, capped_feature_metadata, feature_quality = cap_features_by_quality(
    feature_panel,
    raw_feature_metadata,
    max_features=FEATURE_CONFIG.max_features_per_family,
)

feature_inventory = (
    capped_feature_metadata
    .groupby(["source", "family"])
    .agg(feature_count=("feature", "nunique"))
    .reset_index()
    .sort_values(["source", "family"])
)

event_target_panel, event_target_metadata = build_event_target_panel(
    feature_panel[["symbol", "date"]],
    events,
    TARGET_CONFIG,
)

oracle_target_panel, oracle_target_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols,
    TARGET_CONFIG,
    warehouse=WAREHOUSE,
)

target_panel = combine_target_panels(event_target_panel, oracle_target_panel)
target_metadata = pd.concat([event_target_metadata, oracle_target_metadata], ignore_index=True)
target_summary = summarize_binary_targets(target_panel, target_metadata)

print(
    {
        "event_symbols": len(event_symbols),
        "event_rows": len(events),
        "capped_features": len(selected_features),
        "feature_families": capped_feature_metadata["family"].nunique(),
        "target_rows": len(target_panel),
        "target_columns": target_metadata["target"].nunique(),
        "event_load_seconds": round(event_load_seconds, 3),
        "oracle_seconds": round(oracle_seconds, 3),
    }
)

display(feature_inventory)
display(target_summary.sort_values(["target_family", "positive_rows"], ascending=[True, False]).head(80))

In [ ]:
matrix, training_panel = evaluate_feature_target_matrix(
    feature_panel,
    capped_feature_metadata,
    target_panel,
    target_metadata,
    min_rows=MIN_ROWS_PER_TASK,
    min_positive_rows=MIN_POSITIVE_ROWS_PER_TASK,
    min_feature_coverage=MIN_FEATURE_COVERAGE,
)

usable_matrix = matrix.query("status == 'usable'").copy()
usable_targets = usable_matrix["target"].drop_duplicates().tolist()
usable_feature_families = usable_matrix[["source", "feature_family"]].drop_duplicates()

print(
    {
        "matrix_rows": len(matrix),
        "usable_pairs": len(usable_matrix),
        "training_panel_rows": len(training_panel),
        "usable_targets": len(usable_targets),
        "usable_feature_families": len(usable_feature_families),
        "elapsed_seconds": round(perf_counter() - started, 3),
    }
)

display(usable_matrix.head(80))

## Convert Feature Families to Text Rows

Each model example is long-form:

```text
sample = one (date, symbol, feature_family, target_task)
text   = feature_family=<family> source=<source> feature_a=1.23 feature_b=-4.56 ...
label  = 0 or 1 for that target task
```

`symbol` and `date` stay in the dataframe for splitting and auditing, but are deliberately excluded from the text so the model cannot memorize ticker/date identities.

In [ ]:
def sanitize_task_name(value: str) -> str:
    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    if not text:
        raise ValueError(f"Invalid task name from {value!r}")
    return f"task_{text}"


def format_feature_value(value: object) -> str | None:
    if value is None or pd.isna(value):
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        text = str(value).strip()
        return text if text else None
    if not math.isfinite(number):
        return None
    rounded = round(number, 2)
    if rounded == 0:
        rounded = 0.0
    return f"{rounded:.2f}"


def feature_family_text(row: pd.Series, features: list[str], *, source: str, family: str) -> str:
    pairs = [f"source={source}", f"feature_family={family}"]
    for feature in features:
        value = format_feature_value(row.get(feature))
        if value is not None:
            pairs.append(f"{feature}={value}")
    return " ".join(pairs)


def build_flair_long_frame(
    training_panel: pd.DataFrame,
    feature_metadata: pd.DataFrame,
    matrix: pd.DataFrame,
    *,
    min_feature_coverage: float,
    max_rows_per_task_split: int | None = None,
) -> pd.DataFrame:
    usable = matrix.query("status == 'usable'").copy()
    if usable.empty:
        raise RuntimeError("No usable feature-family/target pairs were found for Flair training.")

    target_to_task = {target: sanitize_task_name(target) for target in usable["target"].drop_duplicates()}
    rows = []
    for (source, family), pair_rows in usable.groupby(["source", "feature_family"], sort=True):
        family_features = (
            feature_metadata
            .loc[
                feature_metadata["source"].eq(source)
                & feature_metadata["family"].eq(family)
                & feature_metadata["feature"].isin(training_panel.columns),
                "feature",
            ]
            .drop_duplicates()
            .tolist()
        )
        if not family_features:
            continue

        feature_values = training_panel[family_features].apply(pd.to_numeric, errors="coerce")
        coverage_mask = feature_values.notna().mean(axis=1).ge(min_feature_coverage)
        base = training_panel.loc[coverage_mask, ["symbol", "date", *family_features]].copy()
        if base.empty:
            continue

        text_values = base.apply(
            lambda row: feature_family_text(row, family_features, source=source, family=family),
            axis=1,
        )
        text_base = pd.DataFrame(
            {
                "symbol": base["symbol"].astype(str).str.upper().to_numpy(),
                "date": pd.to_datetime(base["date"], errors="coerce").dt.normalize().to_numpy(),
                "source": source,
                "feature_family": family,
                "text": text_values.to_numpy(),
            },
            index=base.index,
        )

        for target in pair_rows["target"].drop_duplicates().tolist():
            if target not in training_panel.columns:
                continue
            labels = pd.to_numeric(training_panel.loc[text_base.index, target], errors="coerce").fillna(0).astype("int8")
            task_frame = text_base.copy()
            task_frame["target_task"] = target
            task_frame["task_id"] = target_to_task[target]
            task_frame["label_type"] = target_to_task[target]
            task_frame["label"] = np.where(labels.gt(0), "positive", "negative")
            rows.append(task_frame)

    if not rows:
        raise RuntimeError("Feature-family text conversion produced no training rows.")

    out = pd.concat(rows, ignore_index=True).dropna(subset=["date", "text", "label"])
    out = out.loc[out["text"].str.len().gt(0)].copy()
    out = out.sort_values(["date", "symbol", "feature_family", "target_task"]).reset_index(drop=True)
    if max_rows_per_task_split is not None:
        out = cap_rows_per_task_split(out, max_rows_per_task_split)
    return out


def chronological_split(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    ordered = frame.sort_values(["date", "symbol", "feature_family", "target_task"]).reset_index(drop=True)
    splits = {
        "train": ordered.loc[ordered["date"].le(TRAIN_END)].copy(),
        "dev": ordered.loc[ordered["date"].gt(TRAIN_END) & ordered["date"].le(DEV_END)].copy(),
        "test": ordered.loc[ordered["date"].gt(DEV_END)].copy(),
    }
    if min(len(split) for split in splits.values()) > 0:
        return splits

    first = max(2, int(len(ordered) * 0.70))
    second = max(first + 1, int(len(ordered) * 0.85))
    return {
        "train": ordered.iloc[:first].copy(),
        "dev": ordered.iloc[first:second].copy(),
        "test": ordered.iloc[second:].copy(),
    }


def cap_rows_per_task_split(frame: pd.DataFrame, max_rows: int) -> pd.DataFrame:
    if max_rows <= 0:
        return frame
    pieces = []
    for (_, _), group in frame.groupby(["split", "task_id"], sort=False):
        if len(group) <= max_rows:
            pieces.append(group)
        else:
            positives = group.loc[group["label"].eq("positive")]
            negatives = group.loc[group["label"].ne("positive")]
            pos_n = min(len(positives), max_rows // 2)
            neg_n = min(len(negatives), max_rows - pos_n)
            sampled = pd.concat(
                [
                    positives.sample(n=pos_n, random_state=RANDOM_SEED) if pos_n else positives.head(0),
                    negatives.sample(n=neg_n, random_state=RANDOM_SEED) if neg_n else negatives.head(0),
                ],
                ignore_index=True,
            )
            if len(sampled) < max_rows:
                remainder = group.drop(index=sampled.index, errors="ignore")
                sampled = pd.concat(
                    [sampled, remainder.sample(n=min(len(remainder), max_rows - len(sampled)), random_state=RANDOM_SEED)],
                    ignore_index=True,
                )
            pieces.append(sampled)
    return pd.concat(pieces, ignore_index=True).sort_values(["date", "symbol", "feature_family", "target_task"]).reset_index(drop=True)

In [ ]:
long_frame = build_flair_long_frame(
    training_panel,
    capped_feature_metadata,
    usable_matrix,
    min_feature_coverage=MIN_FEATURE_COVERAGE,
    max_rows_per_task_split=None,
)

splits = chronological_split(long_frame)
for split_name, split_frame in splits.items():
    split_frame["split"] = split_name
long_frame = pd.concat(splits.values(), ignore_index=True)
if MAX_ROWS_PER_TASK_SPLIT is not None:
    long_frame = cap_rows_per_task_split(long_frame, MAX_ROWS_PER_TASK_SPLIT)
    splits = {name: frame.drop(columns=["split"], errors="ignore") for name, frame in long_frame.groupby("split", sort=False)}
else:
    splits = {name: frame.drop(columns=["split"], errors="ignore") for name, frame in long_frame.groupby("split", sort=False)}

text_audit = (
    long_frame.assign(token_estimate=long_frame["text"].str.split().str.len())
    .groupby(["split", "task_id"])
    .agg(
        rows=("label", "size"),
        positives=("label", lambda s: int((s == "positive").sum())),
        positive_rate=("label", lambda s: float((s == "positive").mean())),
        feature_families=("feature_family", "nunique"),
        median_token_estimate=("token_estimate", "median"),
        max_token_estimate=("token_estimate", "max"),
    )
    .reset_index()
    .sort_values(["task_id", "split"])
)

print(
    {
        "long_rows": len(long_frame),
        "tasks": long_frame["task_id"].nunique(),
        "feature_families": long_frame["feature_family"].nunique(),
        "symbols": long_frame["symbol"].nunique(),
        "min_date": long_frame["date"].min(),
        "max_date": long_frame["date"].max(),
    }
)

display(text_audit.head(120))
display(long_frame[["date", "symbol", "feature_family", "target_task", "split", "source", "task_id", "label", "text"]].head(10))

assert not long_frame["text"].str.contains(r"\bsymbol=|\bdate=", regex=True).any(), "symbol/date leaked into text"

## Build a Flair Native Multitask Corpus

Each target column is a separate Flair task. The same feature-family text can appear under multiple tasks, but each row carries exactly one `multitask_id` and one binary label for that target.

In [ ]:
def frame_to_multitask_sentences(frame: pd.DataFrame) -> list[Sentence]:
    sentences: list[Sentence] = []
    for row in frame.itertuples(index=False):
        sentence = Sentence(str(row.text))
        sentence.add_label(str(row.label_type), str(row.label))
        sentence.add_label("multitask_id", str(row.task_id))
        sentences.append(sentence)
    return sentences


def build_multitask_corpus(splits: dict[str, pd.DataFrame]) -> Corpus:
    return Corpus(
        train=frame_to_multitask_sentences(splits["train"]),
        dev=frame_to_multitask_sentences(splits["dev"]),
        test=frame_to_multitask_sentences(splits["test"]),
        sample_missing_splits=False,
    )


def build_binary_multitask_model(corpus: Corpus, task_ids: list[str]) -> MultitaskModel:
    embeddings = TransformerDocumentEmbeddings(
        FLAIR_TRANSFORMER,
        fine_tune=FINE_TUNE_TRANSFORMER,
        layers="-1",
        layer_mean=False,
        allow_long_sentences=True,
    )
    classifiers = []
    for task_id in task_ids:
        label_dictionary = corpus.make_label_dictionary(task_id, add_unk=False)
        classifier = TextClassifier(embeddings, label_type=task_id, label_dictionary=label_dictionary)
        classifiers.append(classifier)
    model = MultitaskModel(
        classifiers,
        task_ids=task_ids,
        loss_factors=[1.0 for _ in task_ids],
        use_all_tasks=False,
    )
    return model.to(flair.device)


def predict_task_classifier(model: MultitaskModel, task_id: str, rows: pd.DataFrame) -> np.ndarray:
    task = model.tasks[task_id]
    sentences = [Sentence(str(text)) for text in rows["text"].tolist()]
    task.predict(sentences, label_name="pred", mini_batch_size=FLAIR_EVAL_BATCH_SIZE)
    return np.array([sentence.get_labels("pred")[0].value for sentence in sentences])


def evaluate_multitask_model(model: MultitaskModel, frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for task_id, task_rows in frame.groupby("task_id", sort=True):
        y_true = task_rows["label"].astype(str).to_numpy()
        if len(np.unique(y_true)) < 2:
            rows.append(
                {
                    "task_id": task_id,
                    "rows": len(task_rows),
                    "positives": int((task_rows["label"] == "positive").sum()),
                    "accuracy": np.nan,
                    "macro_f1": np.nan,
                    "precision_positive": np.nan,
                    "recall_positive": np.nan,
                    "note": "single_class_test_set",
                }
            )
            continue
        y_pred = predict_task_classifier(model, task_id, task_rows)
        rows.append(
            {
                "task_id": task_id,
                "target_task": task_rows["target_task"].iloc[0],
                "rows": len(task_rows),
                "positives": int((task_rows["label"] == "positive").sum()),
                "positive_rate": float((task_rows["label"] == "positive").mean()),
                "accuracy": accuracy_score(y_true, y_pred),
                "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
                "precision_positive": precision_score(y_true, y_pred, labels=["positive"], average="macro", zero_division=0),
                "recall_positive": recall_score(y_true, y_pred, labels=["positive"], average="macro", zero_division=0),
                "note": "ok",
            }
        )
    return pd.DataFrame(rows).sort_values(["macro_f1", "positives"], ascending=[False, False]).reset_index(drop=True)

In [ ]:
# Execute this cell when you are ready to train.
# It is intentionally separate from dataset construction so you can inspect `long_frame` first.

train_started = perf_counter()
corpus = build_multitask_corpus(splits)
task_ids = sorted(long_frame["task_id"].drop_duplicates().tolist())
model = build_binary_multitask_model(corpus, task_ids)
trainer = ModelTrainer(model, corpus)

train_result = trainer.fine_tune(
    base_path=ARTIFACT_DIR,
    learning_rate=FLAIR_LEARNING_RATE,
    mini_batch_size=FLAIR_BATCH_SIZE,
    mini_batch_chunk_size=FLAIR_BATCH_CHUNK_SIZE,
    eval_batch_size=FLAIR_EVAL_BATCH_SIZE,
    max_epochs=FLAIR_EPOCHS,
    embeddings_storage_mode="none",
    save_final_model=False,
    create_file_logs=True,
    create_loss_file=True,
)

train_seconds = perf_counter() - train_started
print({"tasks": len(task_ids), "train_seconds": round(train_seconds, 3), "artifact_dir": str(ARTIFACT_DIR)})

In [ ]:
# Execute after training to inspect task-level results.

test_results = evaluate_multitask_model(model, splits["test"])
display(test_results)

display(
    Markdown(
        "
".join(
            [
                "## Training Notes",
                "",
                f"- Universe: {len(symbols)} symbols from FMP with market cap >= ${MIN_MARKET_CAP:,.0f}.",
                f"- Long-form text rows: {len(long_frame):,} across {long_frame['feature_family'].nunique()} feature families and {long_frame['task_id'].nunique()} target tasks.",
                "- Text excludes symbol and date by assertion; they are metadata only for splitting and auditing.",
                "- Numeric feature values are rounded to two decimals before string formatting.",
                "- Treat this as a trainability smoke test first. Sparse event tasks can show unstable metrics even when the notebook runs correctly.",
            ]
        )
    )
)